## Bijdrage Raoul Buffels — DeepLearningTeam10

Deze Citi Bike-uitwerking is aangeleverd als de bijdrage van **Raoul Buffels**. Teamleden: Jorre Van Dyck, Milan Wouters, Raoul Buffels en Andrew Noeyens. De concrete menselijke taakverdeling moet het team zelf aanvullen. **AI-gebruik:** Codex hielp met implementatie, uitleg, tests, uitvoering en verpakking.

De codecellen zijn uitgevoerd in de oorspronkelijke lokale `Citybike_full`-omgeving op 9 oktober 2026. Deze export behoudt die echte uitvoer; alleen deze Markdown-cel is toegevoegd. Lokale paden, timings en fingerprints in de rapporten beschrijven de oorspronkelijke uitvoering. Start voor heruitvoering vanuit `RaoulBuffels` of de map `notebooks` met de bijbehorende Python 3.11-omgeving. AWS is een afzonderlijke latere taak.

# Baseline en Ridge

Bijdragen: door team in te vullen. De training gebeurt offline via `run.py train`. Dit notebook bewaart echte scores en configuraties; geen eindtest wordt voor tuning gebruikt.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
locations = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'Citybike_full']
ROOT = next((p for p in locations if (p / 'citibike').exists()), None)
if ROOT is None: raise RuntimeError('Start dit notebook vanuit Citybike_full of notebooks.')
sys.path.insert(0, str(ROOT))
from citibike.config import REPORTS, DATA, MODELS
from citibike.datasets import manifest
from citibike.plots import show
def report(name):
    return json.loads((REPORTS / name).read_text(encoding='utf-8'))
def table(name):
    display(pd.read_csv(REPORTS / name, dtype={'start_station_id':'string','end_station_id':'string','station_id':'string'}))


De kern van iedere trainingsstap is hieronder weergegeven. De CLI herhaalt dit voor dezelfde opgeslagen folds, fit preprocessing alleen op de trainingsrijen en bewaart de werkelijke scores. Dit rapportnotebook voert na de eindtest geen nieuwe selectie uit.

```python
fitted = clone(candidate).fit(training[FEATURES], training.rides)
predicted = fitted.predict(validation[FEATURES])
scores = metrics(validation.rides, predicted)
```

In [2]:
from citibike.modeling import candidates
from citibike.features import FEATURES
import joblib
for name in ['seasonal_naive', 'ridge', 'ridge_calendar', 'ridge_no_14d']:
    if (MODELS/(name+'.joblib')).exists():
        model=joblib.load(MODELS/(name+'.joblib'))
        display(Markdown('**Opgeslagen estimator: '+name+'**'))
        display(Markdown(str(model.estimator_)))

**Opgeslagen estimator: seasonal_naive**

SeasonalNaive()

**Opgeslagen estimator: ridge**

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('ridge', Ridge(alpha=10))])

**Opgeslagen estimator: ridge_calendar**

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('ridge', Ridge(alpha=10))])

**Opgeslagen estimator: ridge_no_14d**

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('ridge', Ridge(alpha=10))])

Dezelfde chronologische validatie en operationele voorspelfunctie maken de vergelijking eerlijk.

In [3]:
comparison=pd.read_csv(REPORTS/'model_comparison.csv')
display(comparison[comparison.name.isin(['seasonal_naive', 'ridge', 'ridge_calendar', 'ridge_no_14d'])])

,name,RMSE,MAE,R2,train_RMSE,train_MAE,train_R2,cv_RMSE,fit_cv_seconds
0,seasonal_naive,2826.164920,1510.127732,0.668076,1309.747668,677.011659,0.798089,2086.672928,0.229516
1,ridge,1982.440036,1248.058114,0.836678,1021.691347,578.390365,0.877136,1594.314667,0.866120
6,ridge_calendar,5315.257689,3986.439396,-0.174067,2325.245735,1617.036124,0.363611,2590.232107,0.438945
7,ridge_no_14d,2174.787271,1376.786629,0.803448,1060.539908,606.544454,0.867615,1557.951926,0.503810


Een lagere trainingsfout kan overfit betekenen. Kies op validatie en bekijk de CV-spreiding en looptijd.

In [4]:
records=report('model_comparison.json')
for record in records:
    if record['name'] in ['seasonal_naive', 'ridge', 'ridge_calendar', 'ridge_no_14d']:
        display(Markdown('**'+record['name']+'**'))
        display(pd.DataFrame(record['cv']))
        display(Markdown(str(record['parameters'])))

**seasonal_naive**

,fold,RMSE,MAE,R2,seconds
0,1,1979.271967,1259.720238,0.146485,0.029741
1,2,1797.007685,1250.418155,0.337800,0.031044
2,3,1474.318125,965.205357,0.285118,0.035261
3,4,2565.008427,1614.552083,0.457230,0.036382
4,5,2617.758436,1660.016369,0.626783,0.038523


{'columns': None, 'estimator': 'SeasonalNaive()'}

**ridge**

,fold,RMSE,MAE,R2,seconds
0,1,1410.358708,884.905921,0.566630,0.121878
1,2,1346.493144,883.285060,0.628210,0.131189
2,3,1176.981053,814.124750,0.544393,0.136288
3,4,1984.786066,1254.158220,0.675013,0.129334
4,5,2052.954364,1291.675370,0.770458,0.153673


...

,fold,RMSE,MAE,R2,seconds
0,1,1674.535138,1107.631297,0.389074,0.112595
1,2,1937.044243,1318.133977,0.230571,0.057944
2,3,1395.444717,982.618884,0.359562,0.059053
3,4,3485.826604,2433.370994,-0.002421,0.058408
4,5,4458.309832,3182.851477,-0.082538,0.060284


{'columns': ['hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'year_sin', 'year_cos', 'is_weekend', 'clock_hours'], 'estimator__memory': None, 'estimator__steps': [['standardscaler', 'StandardScaler()'], ['ridge', 'Ridge(alpha=10)']], 'estimator__verbose': False, 'estimator__standardscaler': 'StandardScaler()', 'estimator__ridge': 'Ridge(alpha=10)', 'estimator__standardscaler__copy': True, 'estimator__standardscaler__with_mean': True, 'estimator__standardscaler__with_std': True, 'estimator__ridge__alpha': 10, 'estimator__ridge__copy_X': True, 'estimator__ridge__fit_intercept': True, 'estimator__ridge__max_iter': None, 'estimator__ridge__positive': False, 'estimator__ridge__random_state': None, 'estimator__ridge__solver': 'auto', 'estimator__ridge__tol': 0.0001, 'estimator': "Pipeline(steps=[('standardscaler', StandardScaler()),\n                ('ridge', Ridge(alpha=10))])"}

**ridge_no_14d**

,fold,RMSE,MAE,R2,seconds
0,1,1377.005274,871.687706,0.586885,0.094437
1,2,1178.267580,845.175477,0.715307,0.096270
2,3,1176.106068,789.552901,0.545070,0.101209
3,4,1846.263113,1224.848383,0.718793,0.091279
4,5,2212.117597,1428.963653,0.733486,0.050353


{'columns': ['hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'year_sin', 'year_cos', 'is_weekend', 'clock_hours', 'lag_1d', 'lag_2d', 'lag_7d', 'same_hour_mean_7d', 'previous_day_total'], 'estimator__memory': None, 'estimator__steps': [['standardscaler', 'StandardScaler()'], ['ridge', 'Ridge(alpha=10)']], 'estimator__verbose': False, 'estimator__standardscaler': 'StandardScaler()', 'estimator__ridge': 'Ridge(alpha=10)', 'estimator__standardscaler__copy': True, 'estimator__standardscaler__with_mean': True, 'estimator__standardscaler__with_std': True, 'estimator__ridge__alpha': 10, 'estimator__ridge__copy_X': True, 'estimator__ridge__fit_intercept': True, 'estimator__ridge__max_iter': None, 'estimator__ridge__positive': False, 'estimator__ridge__random_state': None, 'estimator__ridge__solver': 'auto', 'estimator__ridge__tol': 0.0001, 'estimator': "Pipeline(steps=[('standardscaler', StandardScaler()),\n                ('ridge', Ridge(alpha=10))])"}

In [5]:
for path in sorted(REPORTS.glob('*_tuning.csv')):
    if path.stem.split('_tuning')[0] in ['ridge', 'ridge_calendar', 'ridge_no_14d', 'seasonal_naive']:
        display(pd.read_csv(path)[['configuration','cv_RMSE','parameters']])